In [3]:
# =============================================================================
# NOTEBOOK: 04_five_axis_validation.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — A Domain-Agnostic Three-Agent Pipeline (DSRM Design Artifact)
#
# ★ CORE CONTRIBUTION — The Five-Axis Validation Framework
#
#   The pipeline's output is a judgment-type artifact for which no single ground
#   truth is immediately observable. This notebook OPERATIONALIZES the five-axis
#   framework the paper proposes, computing each axis from evidence the three
#   agents already emitted:
#
#     1. Accuracy     : agent output vs. (partial) expert ground truth
#                       — grade agreement + Cohen's kappa; time MAE / MAPE,
#                         computed only over LLM-matched activity pairs.
#     2. Reliability  : Self-Consistency dispersion (Agent-2 rollout CV).
#     3. Efficiency   : API cost & runtime vs. the labor value produced.
#     4. Transparency : rationale coverage (Agent-1) + source tags (Agent-2).
#     5. Robustness   : clarifying-question rate + [MISSING]/prior handling.
#
#   The paper's stance: this notebook DESIGNS and DEMONSTRATES the protocol on
#   one illustrative case; the adequacy thresholds per axis are deliberately
#   left as future standardization work.
#
# Inputs:
#   artifacts/inference/agent1_work_units.json
#   artifacts/inference/agent2_time.json
#   artifacts/inference/agent3_roi.json
#   data/raw/ground_truth_tcb.json        (partial expert reference)
# Output:
#   artifacts/inference/five_axis_report.json
#   artifacts/tables/five_axis_summary.csv
#
# All example content and code are in English for journal submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Bootstrap foundation from Notebook 00 (self-contained)
# =============================================================================
import os
import re
import json
import time
import hashlib
import statistics
import warnings
from pathlib import Path
from typing import Optional, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_RAW  = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "artifacts"
INFER     = ARTIFACTS / "inference"
TAB       = ARTIFACTS / "tables"
CACHE     = ARTIFACTS / "llm_cache"
for p in (INFER, TAB, CACHE):
    p.mkdir(parents=True, exist_ok=True)


def rel(p) -> str:
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


SEED = 42
np.random.seed(SEED)

RUN_MANIFEST = ARTIFACTS / "run_manifest.json"
manifest = json.loads(RUN_MANIFEST.read_text(encoding="utf-8"))
MODELS       = manifest["models"]
DEFAULT_TIER = manifest["default_tier"]
MISSING      = manifest["missing_sentinel"]

load_dotenv(ROOT / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env.")
client = OpenAI(api_key=OPENAI_API_KEY)


# %%
# =============================================================================
# Cell 2. Re-declare cost tracker + llm_call() (identical to nb 00)
# =============================================================================
class CostTracker:
    def __init__(self):
        self.records: list[dict] = []

    def add(self, tier, model, usage, tag=""):
        p = MODELS.get(tier, {})
        pin, pcached, pout = p.get("in"), p.get("cached_in"), p.get("out")
        pt = getattr(usage, "prompt_tokens", 0) or 0
        ct = getattr(usage, "completion_tokens", 0) or 0
        cached = 0
        det = getattr(usage, "prompt_tokens_details", None)
        if det is not None:
            cached = getattr(det, "cached_tokens", 0) or 0
        fresh = max(pt - cached, 0)
        cost = None
        if None not in (pin, pout):
            pc = pcached if pcached is not None else pin
            cost = (fresh * pin + cached * pc + ct * pout) / 1_000_000
        self.records.append({"tag": tag, "tier": tier, "model": model,
                             "prompt_tokens": pt, "cached_tokens": cached,
                             "completion_tokens": ct, "cost_usd": cost})
        return cost or 0.0

    def total_usd(self):
        return float(sum(r["cost_usd"] or 0.0 for r in self.records))

    def flush(self, stage: str):
        """Append this notebook's spend to a shared ledger so the five-axis
        Efficiency computation and the dashboard can read the TRUE cumulative
        pipeline cost across all notebooks (they do not share memory)."""
        ledger = ARTIFACTS / "cost_ledger.json"
        data = json.loads(ledger.read_text(encoding="utf-8")) if ledger.exists() else {}
        data[stage] = {
            "total_usd": round(self.total_usd(), 6),
            "n_calls": len(self.records),
        }
        ledger.write_text(json.dumps(data, ensure_ascii=False, indent=2),
                          encoding="utf-8")
        return data

COST = CostTracker()


def _safe_json(text):
    if not text:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    for opener, closer in (("{", "}"), ("[", "]")):
        i, j = t.find(opener), t.rfind(closer)
        if 0 <= i < j:
            try:
                return json.loads(t[i:j + 1])
            except Exception:
                continue
    return None


def _cache_key(model, system, user, temperature, response_json):
    raw = json.dumps({"m": model, "s": system, "u": user,
                      "t": temperature, "j": response_json},
                     ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def llm_call(system, user, tier=DEFAULT_TIER, temperature=0.0,
             response_json=True, tag="", use_cache=True, max_retries=3):
    model = MODELS[tier]["name"]
    key = _cache_key(model, system, user, temperature, response_json)
    cache_file = CACHE / f"{key}.json"
    if use_cache and cache_file.exists():
        c = json.loads(cache_file.read_text(encoding="utf-8"))
        c["cached"] = True
        # Preserve the ORIGINAL (already-paid) cost so the ledger reflects the
        # true analysis cost even on cached re-runs; also re-log it to COST so
        # cumulative spend is complete regardless of cache state.
        original_cost = c.get("cost_usd", 0.0) or 0.0
        COST.records.append({
            "tag": tag or c.get("model", ""), "tier": c.get("tier", tier),
            "model": c.get("model", ""), "prompt_tokens": 0,
            "cached_tokens": 0, "completion_tokens": 0,
            "cost_usd": original_cost,
        })
        c["cost_usd"] = original_cost
        return c
    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
    }
    if response_json:
        kwargs["response_format"] = {"type": "json_object"}
    kwargs["temperature"] = temperature
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            parsed = _safe_json(text) if response_json else None
            if response_json and parsed is None:
                raise ValueError("Response was not valid JSON.")
            cost = COST.add(tier, model, resp.usage, tag=tag or model)
            out = {"text": text, "json": parsed, "cached": False,
                   "tier": tier, "model": model, "cost_usd": cost}
            if use_cache:
                cache_file.write_text(json.dumps(out, ensure_ascii=False),
                                      encoding="utf-8")
            return out
        except TypeError as e:
            if "temperature" in kwargs:
                kwargs.pop("temperature", None); last_err = e; continue
            last_err = e
        except Exception as e:
            last_err = e; time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"[llm_call] failed after {max_retries} retries: {last_err}")


print("[INFO] CostTracker, llm_call() re-established for nb 04.")


# %%
# =============================================================================
# Cell 3. Load all pipeline artifacts + the partial ground truth
# =============================================================================
A1 = json.loads((INFER / "agent1_work_units.json").read_text(encoding="utf-8"))
A2 = json.loads((INFER / "agent2_time.json").read_text(encoding="utf-8"))
A3 = json.loads((INFER / "agent3_roi.json").read_text(encoding="utf-8"))

GT_PATH = DATA_RAW / "ground_truth_tcb.json"
GROUND_TRUTH = json.loads(GT_PATH.read_text(encoding="utf-8")) if GT_PATH.exists() else None

units = A1["work_units"]
estimates = {e["id"]: e for e in A2["estimates"]}
print(f"[INFO] Agent-1 units: {len(units)}, Agent-2 estimates: {len(estimates)}")
print(f"[INFO] Ground truth: "
      f"{len(GROUND_TRUTH) if GROUND_TRUTH else 0} expert activities "
      f"{'(Accuracy axis ENABLED)' if GROUND_TRUTH else '(Accuracy axis SKIPPED)'}")


# %%
# =============================================================================
# Cell 4. AXIS 1 — ACCURACY (against the partial expert reference)
#
# Challenge: the reference covers only 13 activities and uses different wording
# than the agent's 46 extracted units. We therefore first ask the LLM to MATCH
# each ground-truth activity to at most one extracted unit by meaning, then
# compute agreement ONLY over confident matches. Unmatched reference items are
# reported as recall gaps — an honest, explicit limitation, not hidden.
# =============================================================================
def build_matcher_payload():
    gt = [{"gt_id": g["id"], "name": g["name"]} for g in GROUND_TRUTH]
    ex = [{"unit_id": u["id"], "name": u["name"]} for u in units]
    return gt, ex

accuracy = {"enabled": bool(GROUND_TRUTH)}
if GROUND_TRUTH:
    gt_list, ex_list = build_matcher_payload()

    MATCH_SYSTEM = """\
You align expert-reference activities to pipeline-extracted work units by MEANING
(not by exact wording). For each reference activity, choose the single best
matching extracted unit, or null if none clearly corresponds.

Return ONLY JSON:
{ "matches": [ {"gt_id": "...", "unit_id": "... or null", "confidence": 0..1} ] }
Rules: each unit_id used at most once; use null when no unit is a clear match.
"""
    MATCH_USER = ("Reference activities:\n" + json.dumps(gt_list, ensure_ascii=False)
                  + "\n\nExtracted units:\n" + json.dumps(ex_list, ensure_ascii=False))

    mres = llm_call(MATCH_SYSTEM, MATCH_USER, tier=DEFAULT_TIER, temperature=0.0,
                    tag="axis1_matcher")
    matches = (mres["json"] or {}).get("matches", [])
    # Keep confident, non-null matches.
    good = [m for m in matches
            if m.get("unit_id") and m.get("confidence", 0) >= 0.5]
    gt_by_id = {g["id"]: g for g in GROUND_TRUTH}
    unit_by_id = {u["id"]: u for u in units}
    est_by_id = estimates

    # --- Grade agreement + Cohen's kappa over matched pairs ------------------
    y_true, y_pred, mins_true, mins_pred = [], [], [], []
    pair_rows = []
    for m in good:
        g = gt_by_id.get(m["gt_id"]); u = unit_by_id.get(m["unit_id"])
        if not g or not u:
            continue
        y_true.append(g["auto"]); y_pred.append(u.get("auto_grade", MISSING))
        e = est_by_id.get(u["id"], {})
        mpc = e.get("minutes_per_case")
        if isinstance(mpc, (int, float)) and g.get("min"):
            mins_true.append(float(g["min"])); mins_pred.append(float(mpc))
        pair_rows.append({
            "gt_id": g["id"], "gt_name": g["name"], "unit_id": u["id"],
            "gt_grade": g["auto"], "pred_grade": u.get("auto_grade"),
            "gt_min": g.get("min"), "pred_min": mpc,
        })

    def cohen_kappa(a, b):
        cats = sorted(set(a) | set(b))
        idx = {c: i for i, c in enumerate(cats)}
        n = len(a)
        if n == 0:
            return None
        po = sum(1 for x, y in zip(a, b) if x == y) / n
        # Expected agreement.
        ca = [0] * len(cats); cb = [0] * len(cats)
        for x in a: ca[idx[x]] += 1
        for y in b: cb[idx[y]] += 1
        pe = sum((ca[i]/n) * (cb[i]/n) for i in range(len(cats)))
        return (po - pe) / (1 - pe) if (1 - pe) else None

    grade_agree = (sum(1 for x, y in zip(y_true, y_pred) if x == y) / len(y_true)
                   if y_true else None)
    kappa = cohen_kappa(y_true, y_pred) if y_true else None

    def mae(a, b):  return float(np.mean(np.abs(np.array(a) - np.array(b)))) if a else None
    def mape(a, b):
        a, b = np.array(a, float), np.array(b, float)
        return float(np.mean(np.abs((a - b) / a)) * 100) if len(a) else None

    accuracy.update({
        "matched_pairs": len(good),
        "unmatched_reference": len(GROUND_TRUTH) - len(good),  # recall gap
        "grade_agreement": round(grade_agree, 3) if grade_agree is not None else None,
        "grade_cohen_kappa": round(kappa, 3) if kappa is not None else None,
        "time_MAE_min": round(mae(mins_true, mins_pred), 2) if mins_true else None,
        "time_MAPE_pct": round(mape(mins_true, mins_pred), 1) if mins_true else None,
        "n_time_pairs": len(mins_true),
    })
    print("[AXIS 1 · Accuracy]")
    print(f"   matched pairs        : {accuracy['matched_pairs']} "
          f"(unmatched ref: {accuracy['unmatched_reference']})")
    print(f"   grade agreement      : {accuracy['grade_agreement']}")
    print(f"   grade Cohen's kappa  : {accuracy['grade_cohen_kappa']}")
    print(f"   time MAE (min)       : {accuracy['time_MAE_min']}")
    print(f"   time MAPE (%)        : {accuracy['time_MAPE_pct']}  "
          f"(n={accuracy['n_time_pairs']})")
else:
    print("[AXIS 1 · Accuracy] SKIPPED (no ground truth).")


# %%
# =============================================================================
# Cell 5. AXIS 2 — RELIABILITY (Self-Consistency dispersion from Agent 2)
# =============================================================================
cvs = [e.get("cv_minutes", 0.0) for e in A2["estimates"]]
n_samples = [e.get("n_samples", 0) for e in A2["estimates"]]
reliability = {
    "n_rollouts": A2["n_rollouts"],
    "mean_cv_minutes": round(float(np.mean(cvs)), 3),
    "median_cv_minutes": round(float(np.median(cvs)), 3),
    "pct_nodes_low_cv": round(float(np.mean([c <= 0.25 for c in cvs]) * 100), 1),
    "pct_nodes_full_sampled":
        round(float(np.mean([s == A2["n_rollouts"] for s in n_samples]) * 100), 1),
}
print("[AXIS 2 · Reliability]")
print(f"   rollouts              : {reliability['n_rollouts']}")
print(f"   mean CV (minutes)     : {reliability['mean_cv_minutes']}")
print(f"   nodes with CV<=0.25   : {reliability['pct_nodes_low_cv']}%")


# %%
# =============================================================================
# Cell 6. AXIS 3 — EFFICIENCY (pipeline cost vs. labor value produced)
#
# The true cumulative LLM cost is read from the shared cost ledger that every
# notebook appends to via COST.flush(). If the ledger is absent (e.g. a cached
# re-run with no fresh calls), we fall back to a small observed constant and say
# so. The axis reports a ONE-TIME analysis cost against a RECURRING annual
# saving, so we present both the raw ratio and a plain-language framing rather
# than leaning on a single dramatic number.
# =============================================================================
annual_saving = A3["baseline"]["annual_saving_usd"]

LEDGER = ARTIFACTS / "cost_ledger.json"
if LEDGER.exists():
    ledger = json.loads(LEDGER.read_text(encoding="utf-8"))
    pipeline_cost = round(sum(v.get("total_usd", 0.0) for v in ledger.values()), 6)
    total_calls = int(sum(v.get("n_calls", 0) for v in ledger.values()))
    cost_source = "measured (cost_ledger.json across all notebooks)"
else:
    ledger = {}
    pipeline_cost = 0.02   # conservative observed floor if ledger not built yet
    total_calls = None
    cost_source = "fallback constant (run notebooks with COST.flush to measure)"

# Guard against a zero/near-zero denominator producing an absurd ratio.
ratio = round(annual_saving / pipeline_cost, 0) if pipeline_cost > 1e-6 else None

efficiency = {
    "annual_saving_usd": round(annual_saving, 0),
    "pipeline_llm_cost_usd": pipeline_cost,
    "pipeline_llm_calls": total_calls,
    "cost_source": cost_source,
    "per_stage_cost": {k: v.get("total_usd") for k, v in ledger.items()},
    "saving_to_cost_ratio": ratio,
    "framing": "A one-time analysis costing cents produces an estimated "
               "recurring annual labor saving in the five figures; the axis "
               "measures order-of-magnitude leverage, not a precise multiple.",
}
print("[AXIS 3 · Efficiency]")
print(f"   annual saving (USD)   : ${efficiency['annual_saving_usd']:,.0f}")
print(f"   pipeline LLM cost     : ${efficiency['pipeline_llm_cost_usd']:.4f} "
      f"({cost_source})")
if efficiency["per_stage_cost"]:
    print(f"   per-stage cost        : {efficiency['per_stage_cost']}")
print(f"   saving : cost ratio   : "
      f"{efficiency['saving_to_cost_ratio']:,.0f} : 1"
      if ratio else "   saving : cost ratio   : n/a")


# %%
# =============================================================================
# Cell 7. AXIS 4 — TRANSPARENCY (rationale coverage + source-tag provenance)
# =============================================================================
rationale_present = [bool((u.get("auto_rationale") or "").strip()) for u in units]
src_dist = {}
for e in A2["estimates"]:
    s = e.get("source", "prior")
    src_dist[s] = src_dist.get(s, 0) + 1
transparency = {
    "grade_rationale_coverage_pct":
        round(float(np.mean(rationale_present) * 100), 1),
    "time_source_distribution": src_dist,
    "pct_estimates_grounded":   # implied or stated (i.e. not bare prior)
        round(float(sum(v for k, v in src_dist.items() if k in ("stated", "implied"))
                    / max(sum(src_dist.values()), 1) * 100), 1),
}
print("[AXIS 4 · Transparency]")
print(f"   grade rationale coverage : {transparency['grade_rationale_coverage_pct']}%")
print(f"   time source distribution : {transparency['time_source_distribution']}")
print(f"   grounded (not bare prior): {transparency['pct_estimates_grounded']}%")


# %%
# =============================================================================
# Cell 8. AXIS 5 — ROBUSTNESS (clarifying questions + unknown handling)
# =============================================================================
clar = A2.get("clarifying_questions", [])
missing_actor = sum(1 for u in units if (u.get("actor") == MISSING))
missing_system = sum(1 for u in units if (u.get("system") == MISSING))
robustness = {
    "clarifying_questions": len(clar),
    "clarifying_rate_pct": round(len(clar) / max(len(units), 1) * 100, 1),
    "missing_actor_flagged": missing_actor,
    "missing_system_flagged": missing_system,
    "note": "The interview stated no explicit times; the pipeline surfaced "
            "low-confidence high-impact nodes as questions instead of "
            "silently trusting weak estimates.",
}
print("[AXIS 5 · Robustness]")
print(f"   clarifying questions   : {robustness['clarifying_questions']} "
      f"({robustness['clarifying_rate_pct']}% of nodes)")
print(f"   [MISSING] actor flagged: {robustness['missing_actor_flagged']}")
print(f"   [MISSING] system flagged: {robustness['missing_system_flagged']}")


# %%
# =============================================================================
# Cell 9. Assemble the five-axis report (design artifact; no pass/fail verdict)
#
# Per the paper, adequacy thresholds are intentionally NOT asserted here; the
# framework reports evidence per axis and leaves threshold standardization to
# future confirmatory studies.
# =============================================================================
report = {
    "artifact": "five_axis_validation_report",
    "stance": "protocol demonstrated on one illustrative case; thresholds are "
              "left as future standardization work.",
    "axes": {
        "1_accuracy": accuracy,
        "2_reliability": reliability,
        "3_efficiency": efficiency,
        "4_transparency": transparency,
        "5_robustness": robustness,
    },
}
OUT = INFER / "five_axis_report.json"
OUT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[INFO] Five-axis report -> {rel(OUT)}")

# Flat summary table for the paper.
# Guard every formatted value against None so a missing ledger / metric never
# crashes the summary table.
_ratio = efficiency.get("saving_to_cost_ratio")
_ratio_str = f"{_ratio:,.0f}:1" if isinstance(_ratio, (int, float)) else "n/a"

summary_rows = [
    {"axis": "1 Accuracy",
     "key_metric": "grade kappa / time MAPE",
     "value": (f"kappa={accuracy.get('grade_cohen_kappa')}, "
               f"MAPE={accuracy.get('time_MAPE_pct')}%") if GROUND_TRUTH else "n/a"},
    {"axis": "2 Reliability", "key_metric": "mean CV (minutes)",
     "value": reliability["mean_cv_minutes"]},
    {"axis": "3 Efficiency", "key_metric": "saving : cost ratio",
     "value": _ratio_str},
    {"axis": "4 Transparency", "key_metric": "rationale coverage / grounded",
     "value": f"{transparency['grade_rationale_coverage_pct']}% / "
              f"{transparency['pct_estimates_grounded']}%"},
    {"axis": "5 Robustness", "key_metric": "clarifying-question rate",
     "value": f"{robustness['clarifying_rate_pct']}%"},
]
sdf = pd.DataFrame(summary_rows)
scsv = TAB / "five_axis_summary.csv"
sdf.to_csv(scsv, index=False, encoding="utf-8-sig")
print(f"[INFO] Five-axis summary table -> {rel(scsv)}")
with pd.option_context("display.max_colwidth", 60, "display.width", 160):
    print(sdf.to_string(index=False))
print(f"[INFO] Axis-1 matcher spend this run: ${COST.total_usd():.5f}")

[INFO] CostTracker, llm_call() re-established for nb 04.
[INFO] Agent-1 units: 46, Agent-2 estimates: 46
[INFO] Ground truth: 13 expert activities (Accuracy axis ENABLED)
[AXIS 1 · Accuracy]
   matched pairs        : 13 (unmatched ref: 0)
   grade agreement      : 0.692
   grade Cohen's kappa  : 0.548
   time MAE (min)       : 14.38
   time MAPE (%)        : 66.3  (n=13)
[AXIS 2 · Reliability]
   rollouts              : 5
   mean CV (minutes)     : 0.261
   nodes with CV<=0.25   : 34.8%
[AXIS 3 · Efficiency]
   annual saving (USD)   : $149,065
   pipeline LLM cost     : $0.0000 (measured (cost_ledger.json across all notebooks))
   per-stage cost        : {'01_agent1': 0.0}
   saving : cost ratio   : n/a
[AXIS 4 · Transparency]
   grade rationale coverage : 100.0%
   time source distribution : {'prior': 17, 'implied': 29}
   grounded (not bare prior): 63.0%
[AXIS 5 · Robustness]
   clarifying questions   : 3 (6.5% of nodes)
   [MISSING] actor flagged: 0
   [MISSING] system flagged: 39
[